In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

In [3]:
train_df = train_identity.merge(train_transaction)

In [4]:
train_transaction.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
train_transaction.shape

(590540, 394)

In [6]:
train_transaction.isna().sum()

TransactionID          0
isFraud                0
TransactionDT          0
TransactionAmt         0
ProductCD              0
                   ...  
V335              508189
V336              508189
V337              508189
V338              508189
V339              508189
Length: 394, dtype: int64

In [7]:
for i in train_transaction.columns:
    no = train_transaction[i].isna().sum()
    if no > 50000:
        print(i,':',no.sum())
print(f'Highest missing val ;{i}')
    

addr1 : 65706
addr2 : 65706
dist1 : 352271
dist2 : 552913
P_emaildomain : 94456
R_emaildomain : 453249
D2 : 280797
D3 : 262878
D4 : 168922
D5 : 309841
D6 : 517353
D7 : 551623
D8 : 515614
D9 : 515614
D10 : 76022
D11 : 279287
D12 : 525823
D13 : 528588
D14 : 528353
D15 : 89113
M1 : 271100
M2 : 271100
M3 : 271100
M4 : 281444
M5 : 350482
M6 : 169360
M7 : 346265
M8 : 346252
M9 : 346252
V1 : 279287
V2 : 279287
V3 : 279287
V4 : 279287
V5 : 279287
V6 : 279287
V7 : 279287
V8 : 279287
V9 : 279287
V10 : 279287
V11 : 279287
V12 : 76073
V13 : 76073
V14 : 76073
V15 : 76073
V16 : 76073
V17 : 76073
V18 : 76073
V19 : 76073
V20 : 76073
V21 : 76073
V22 : 76073
V23 : 76073
V24 : 76073
V25 : 76073
V26 : 76073
V27 : 76073
V28 : 76073
V29 : 76073
V30 : 76073
V31 : 76073
V32 : 76073
V33 : 76073
V34 : 76073
V35 : 168969
V36 : 168969
V37 : 168969
V38 : 168969
V39 : 168969
V40 : 168969
V41 : 168969
V42 : 168969
V43 : 168969
V44 : 168969
V45 : 168969
V46 : 168969
V47 : 168969
V48 : 168969
V49 : 168969
V50 : 168969

In [8]:
small_df = train_transaction.dropna(subset=['V339'])

In [9]:
small_df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,2987008,0,86535,15.0,H,2803,100.0,150.0,visa,226.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16,2987016,0,86620,30.0,H,1790,555.0,150.0,visa,226.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17,2987017,0,86668,100.0,H,11492,111.0,150.0,mastercard,219.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22,2987022,0,86786,50.0,H,1724,583.0,150.0,visa,226.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
small_df.shape

(82351, 394)

In [11]:
small_df.isna().sum()

TransactionID     0
isFraud           0
TransactionDT     0
TransactionAmt    0
ProductCD         0
                 ..
V335              0
V336              0
V337              0
V338              0
V339              0
Length: 394, dtype: int64

In [12]:
for col in small_df.columns:
    num = small_df[col].isna().sum()
    if num > 0:
        print(col,':',num.sum())

card2 : 397
card3 : 1
card4 : 11
card5 : 583
card6 : 7
addr1 : 343
addr2 : 343
dist1 : 82351
dist2 : 71465
P_emaildomain : 11674
R_emaildomain : 11426
D1 : 41
D2 : 69728
D3 : 77844
D4 : 76131
D5 : 79085
D6 : 73516
D7 : 75937
D8 : 36861
D9 : 36861
D10 : 72272
D11 : 82351
D12 : 82351
D13 : 78347
D14 : 71247
D15 : 72785
M1 : 82351
M2 : 82351
M3 : 82351
M4 : 82351
M5 : 82351
M6 : 82351
M7 : 82351
M8 : 82351
M9 : 82351
V1 : 82351
V2 : 82351
V3 : 82351
V4 : 82351
V5 : 82351
V6 : 82351
V7 : 82351
V8 : 82351
V9 : 82351
V10 : 82351
V11 : 82351
V12 : 72323
V13 : 72323
V14 : 72323
V15 : 72323
V16 : 72323
V17 : 72323
V18 : 72323
V19 : 72323
V20 : 72323
V21 : 72323
V22 : 72323
V23 : 72323
V24 : 72323
V25 : 72323
V26 : 72323
V27 : 72323
V28 : 72323
V29 : 72323
V30 : 72323
V31 : 72323
V32 : 72323
V33 : 72323
V34 : 72323
V35 : 76154
V36 : 76154
V37 : 76154
V38 : 76154
V39 : 76154
V40 : 76154
V41 : 76154
V42 : 76154
V43 : 76154
V44 : 76154
V45 : 76154
V46 : 76154
V47 : 76154
V48 : 76154
V49 : 76154
V50

In [13]:
cols_to_drop = [col for col in small_df.columns if small_df[col].isna().sum() > 15000]
df_small = small_df.drop(columns=cols_to_drop)
df_small.shape

(82351, 275)

In [14]:
for i in df_small.columns:
    print(i, df_small[i].isna().sum())
    

TransactionID 0
isFraud 0
TransactionDT 0
TransactionAmt 0
ProductCD 0
card1 0
card2 397
card3 1
card4 11
card5 583
card6 7
addr1 343
addr2 343
P_emaildomain 11674
R_emaildomain 11426
C1 0
C2 0
C3 0
C4 0
C5 0
C6 0
C7 0
C8 0
C9 0
C10 0
C11 0
C12 0
C13 0
C14 0
D1 41
V95 0
V96 0
V97 0
V98 0
V99 0
V100 0
V101 0
V102 0
V103 0
V104 0
V105 0
V106 0
V107 0
V108 0
V109 0
V110 0
V111 0
V112 0
V113 0
V114 0
V115 0
V116 0
V117 0
V118 0
V119 0
V120 0
V121 0
V122 0
V123 0
V124 0
V125 0
V126 0
V127 0
V128 0
V129 0
V130 0
V131 0
V132 0
V133 0
V134 0
V135 0
V136 0
V137 0
V138 406
V139 406
V140 406
V141 406
V142 406
V143 400
V144 400
V145 400
V146 406
V147 406
V148 406
V149 406
V150 400
V151 400
V152 400
V153 406
V154 406
V155 406
V156 406
V157 406
V158 406
V159 400
V160 400
V161 406
V162 406
V163 406
V164 400
V165 400
V166 400
V167 2389
V168 2389
V169 2417
V170 2417
V171 2417
V172 2389
V173 2389
V174 2417
V175 2417
V176 2389
V177 2389
V178 2389
V179 2389
V180 2417
V181 2389
V182 2389
V183 2389
V184 241

In [15]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import KNNImputer

In [16]:
train_identity.shape

(144233, 41)

In [17]:
train_df.shape

(144233, 434)

In [18]:
train_identity.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


In [19]:
train_df.shape

(144233, 434)

In [20]:
train_df.describe()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
count,1.442330e+05,144233.000000,140872.000000,66324.000000,66324.000000,136865.000000,136865.000000,5155.000000,5155.000000,74926.000000,...,82041.000000,82041.000000,82041.000000,82041.000000,82041.000000,82041.000000,82041.000000,82041.000000,82041.000000,82041.000000
mean,3.236329e+06,-10.170502,174716.584708,0.060189,-0.058938,1.615585,-6.698710,13.285354,-38.600388,0.091023,...,0.777733,723.339755,1379.108414,1017.190218,9.837929,59.213495,28.592672,55.461163,151.546395,100.950114
std,1.788496e+05,14.347949,159651.816856,0.598231,0.701015,5.249856,16.491104,11.384207,26.084899,0.983842,...,4.735065,6222.466950,11181.344838,7964.623789,244.320100,388.035252,275.048519,669.707372,1096.739466,816.354359
min,2.987004e+06,-100.000000,1.000000,-13.000000,-28.000000,-72.000000,-100.000000,-46.000000,-100.000000,-36.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.077142e+06,-10.000000,67992.000000,0.000000,0.000000,0.000000,-6.000000,5.000000,-48.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,3.198818e+06,-5.000000,125800.500000,0.000000,0.000000,0.000000,0.000000,14.000000,-34.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,3.392923e+06,-5.000000,228749.000000,0.000000,0.000000,1.000000,0.000000,22.000000,-23.000000,0.000000,...,0.000000,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,3.577534e+06,0.000000,999595.000000,10.000000,0.000000,52.000000,0.000000,61.000000,0.000000,25.000000,...,55.000000,160000.000000,160000.000000,160000.000000,55125.000000,55125.000000,55125.000000,104060.000000,104060.000000,104060.000000


In [21]:
train_df.isna().sum()

TransactionID        0
id_01                0
id_02             3361
id_03            77909
id_04            77909
                 ...  
V335             62192
V336             62192
V337             62192
V338             62192
V339             62192
Length: 434, dtype: int64